In [1]:
import sys, os
sys.path.append(os.path.abspath('../..'))
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

base_folder = "/data/big_rim/rsync_dcc_sum/Oct3V1" #"/data/big_rim/rsync_dcc_sum/24summ" #"/data/big_rim/rsync_dcc_sum/25Apri_social" #"/data/big_rim/rsync_dcc_sum/Oct3V1" #"/hpc/group/tdunn/Bryan_Rigs/BigOpenField/24summ"  # Replace with your base folder
# save_path = os.path.join(base_folder, 'paret')
failed_paths_file = '/data/big_rim/rsync_dcc_sum/Oct3V1/sync_failed_brws.txt'  # File containing failed paths


force_rescan_rec_files = [
    # ('2023-10-01', '001'),
    # ('2023-10-02', '002'),
    # Add more as needed
]
rescan_threshold_days = 0.0001 # 7 days, but guess if i mess up i can just change it to automatically rescan all, smile... #0.1

log_folder_to_parquet_sep(base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
                            force_rescan_rec_files=force_rescan_rec_files,
                            rescan_threshold_days=rescan_threshold_days)


all_df = read_all_parquet_files(base_folder)

Log for 20241217v1l23re1_p20241217v1l23BE0 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1_p20241217v1l23BE0/folder_log.parquet
Log for 20241217v1l23re1 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_21/20241217v1l23re1/folder_log.parquet
Log for trashmini_240916v1r1_miceleash_test_19_15 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2025_01_23/trashmini_240916v1r1_miceleash_test_19_15/folder_log.parquet
Log for #calib_before saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/#calib_before/folder_log.parquet
Log for 24Anshu_f_paint_2mice_2 saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/24Anshu_f_paint_2mice_2/folder_log.parquet
Log for #Anshu_f_bleach_2mice saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/#Anshu_f_bleach_2mice/folder_log.parquet
Log for 24Anshu_f_paint_2mice saved at /data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_03_micecolor_test/24Anshu_f_paint_2mice/folder_log.parquet
Log for 24Ansh

In [2]:
import pyarrow.compute as pc
from functools import reduce


table = all_df #combined_df
# Filter mir_generate_param == 0 and sync != 3
conditions = [
   # pc.equal(table['mir_generate_param'], '0'),
   # pc.equal(table['sync'], '1'),
   # pc.not_equal(table['sync'], '3'),
   # pc.equal(table['com'], '1'),
   # # pc.equal(table['com_vis'], '1'),
   # # pc.equal(table['v1'], '1'),
   # pc.equal(table['dannce'], '1'),
   #  
   # pc.equal(table['test'], '1'),
   # pc.equal(table['dannce_vis'], '1'),
   pc.equal(table['social'], '1'),  
   pc.equal(table['mini_6cam_map'], '1'),
   # pc.equal(table['date_folder'], '2025_05_16'),
   pc.equal(table['mini_rec_sync_com'], '1'),
   # pc.equal(table['mini_rec_sync'], '1'),
   #mini_rec_sync mini_rec_sync_com
   # mini_6cam_map
]

filter_mask = reduce(pc.and_, conditions)


# Apply the filter and print the results
filtered_table = table.filter(filter_mask)

# Print each row of the filtered table
print(filtered_table.to_pandas())  # This will display the filtered data in a familiar pandas-like format


   mir_generate_param sync mini_6cam_map dropf_handle com com_vis social  \
0                   1    1             1            0   1       1      1   
1                   1    1             1            0   1       1      1   
2                   1    1             1            0   1       1      1   
3                   1    1             1            0   1       1      1   
4                   1    1             1            0   1       1      1   
5                   1    1             1            0   1       1      1   
6                   1    1             1            0   1       1      1   
7                   1    1             1            0   1       1      1   
8                   1    1             1            0   1       1      1   
9                   1    1             1            0   1       1      1   
10                  1    1             1            0   1       1      1   

   miniscope test after_oxytocin before_oxytocin dannce dannce_vis  \
0          1    1

In [3]:
# # export_plan_min.py  (rev: id+time+tiers)
# import os, re, csv, glob, shlex, subprocess, datetime as dt
# import pyarrow as pa
# import pyarrow.parquet as pq

# # -------- helpers --------

# def _truthy(x):
#     return str(x).strip().lower() in {"1","true","t","yes","y"}

# def _to_str(x, default=""):
#     return "" if x is None else str(x)

# def _join(*a):
#     return os.path.join(*[p for p in a if p not in (None, "", ".")])

# def load_aliases(csv_path=None):
#     m = {}
#     if not csv_path or not os.path.exists(csv_path): return m
#     with open(csv_path, newline="", encoding="utf-8") as f:
#         for r in csv.DictReader(f):
#             seen = r.get("seen_name","").strip()
#             canon = r.get("canonical_name","").strip()
#             if seen:
#                 m[seen.lower()] = (canon or seen).strip()
#     return m

# def canonical(name, alias_map):
#     if not name: return "UNK"
#     k = name.strip().lower()
#     return alias_map.get(k, name.strip())

# def _find_path_segment(pattern, parts):
#     for p in parts:
#         if re.fullmatch(pattern, p): return p
#     return None

# def _split_path(path):
#     return [p for p in path.replace("\\","/").split("/") if p]

# def _extract_date_time(row):
#     """
#     Returns ('YY_MM_DD', 'HHMMSS').
#     Date priority: explicit date_folder 'YYYY_MM_DD' -> convert to 'YY_MM_DD'; else from path.
#     Time priority: any 'HH_MM_SS' or 'HH_MM' in fields or session folder (after date).
#                   If only HH_MM is present -> seconds '00'.
#                   Else last-resort: current UTC.
#     """
#     rec_path = _to_str(row.get("rec_path"))
#     parts = _split_path(rec_path)

#     # --- date ---
#     df = _to_str(row.get("date_folder"))
#     m = re.fullmatch(r"(\d{4})_(\d{2})_(\d{2})", df) if df else None
#     if m:
#         yymmdd = f"{m.group(1)[2:]}_{m.group(2)}_{m.group(3)}"
#     else:
#         seg = _find_path_segment(r"\d{4}_\d{2}_\d{2}", parts)
#         if seg:
#             y, mo, d = seg.split("_")
#             yymmdd = f"{y[2:]}_{mo}_{d}"
#         else:
#             now = dt.datetime.utcnow()
#             yymmdd = now.strftime("%y_%m_%d")

#     # --- time ---
#     # consider explicit fields
#     candidates = [
#         _to_str(row.get("time")),
#         _to_str(row.get("time_folder")),
#         _to_str(row.get("rec_file")),
#         _session_dir_name(parts)  # session folder often carries HH_MM
#     ]
#     hms = None
#     for s in candidates:
#         if not s: continue
#         m3 = re.search(r"(\d{2})_(\d{2})_(\d{2})", s)
#         if m3:
#             hms = f"{m3.group(1)}{m3.group(2)}{m3.group(3)}"
#             break
#         m2 = re.search(r"(\d{2})_(\d{2})(?!_)", s)  # HH_MM
#         if m2:
#             hms = f"{m2.group(1)}{m2.group(2)}00"
#             break
#     if not hms:
#         now = dt.datetime.utcnow()
#         hms = now.strftime("%H%M%S")
#     return yymmdd, hms

# def _session_dir_name(parts):
#     """Return the directory *after* YYYY_MM_DD, else last component."""
#     # find first date segment; return next part
#     for i, p in enumerate(parts):
#         if re.fullmatch(r"\d{4}_\d{2}_\d{2}", p):
#             return parts[i+1] if i+1 < len(parts) else parts[-1]
#     return parts[-1] if parts else ""

# # ---- ID inference (don’t use project root) ----

# _ANIMAL_TOKEN_RE = re.compile(
#     r"(?ix)"
#     r"(?:^|[_\- ])"
#     r"(?:"
#     r"v\d+[lre]?\d+[a-z]?"         # v1l5r1, v1re1f, v1r2, v1l5
#     r"|[a-z]{1,3}\d{1,3}[a-z]?\d?" # pmc, re1f, le1, r2 etc
#     r")"
#     r"(?:$|[_\- ])"
# )

# def _tokens_from_session_name(name):
#     # split by separators; filter short noise; find pattern candidates
#     raw = re.split(r"[_\- ]+", name)
#     toks = []
#     for t in raw:
#         if len(t) < 3: continue
#         if _ANIMAL_TOKEN_RE.search(t):
#             toks.append(t)
#     # de-dup while preserving order
#     seen = set(); out=[]
#     for t in toks:
#         tl = t.lower()
#         if tl in seen: continue
#         seen.add(tl); out.append(t)
#     return out

# def _pick_id(row, alias_map, is_social):
#     # 1) use explicit columns if present
#     get = lambda k: row.get(k) if k in row else None
#     if is_social:
#         a = _to_str(get("animal_a"))
#         b = _to_str(get("animal_b"))
#         pair = _to_str(get("pair_id"))
#         cand = [canonical(a, alias_map) for a in (a,b) if a] or None
#         if cand and len(cand) >= 1:
#             return "+".join(sorted(cand))
#         if pair:
#             parts = [canonical(x, alias_map) for x in re.split(r"[+_, ]+", pair) if x]
#             if parts:
#                 return "+".join(sorted(parts))

#     else:
#         a = _to_str(get("animal") or get("animal_id"))
#         if a:
#             return canonical(a, alias_map)

#     # 2) infer from session folder (never from project root)
#     session = _session_dir_name(_split_path(_to_str(get("rec_path"))))
#     candidates = _tokens_from_session_name(session)
#     if candidates:
#         canon = [canonical(c, alias_map) for c in candidates]
#         if is_social:
#             return "+".join(sorted(canon[:2])) if len(canon) >= 2 else canon[0]
#         return canon[0]

#     # 3) last resort
#     return "UNK"

# # ---- tiers ----

# def _assign_tiers(row, include_visuals=False, include_videos=True):
#     has_com   = _truthy(row.get("com"))
#     has_dan   = _truthy(row.get("dannce"))
#     has_align = _truthy(row.get("mini_rec_sync")) or _truthy(row.get("mini_rec_sync_com"))

#     tier0 = ["metaData.json", "calib/"]
#     if include_videos:
#         tier0.insert(1, "videos/")

#     tier1 = []
#     if has_com:   tier1 += ["COM/predict*/com3d*.mat"]
#     if has_dan:   tier1 += ["DANNCE/predict*/save_data_AVG.mat"]
#     if has_align: tier1 += ["MIR_Aligned/only_com*.h5", "MIR_Aligned/aligned_predictions_with_ca*.h5"]

#     tier2 = []
#     if include_visuals:
#         tier2 = [
#             "COM/predict*/vis/*.jpg",
#             "COM/predict*/vis/*.png",
#             "DANNCE/predict*/vis/*.jpg",
#             "DANNCE/predict*/vis/*.png",
#         ]

#     tiers_applied, paths = [], []
#     if tier0: tiers_applied.append("Tier0"); paths += tier0
#     if tier1: tiers_applied.append("Tier1"); paths += tier1
#     if tier2: tiers_applied.append("Tier2"); paths += tier2
#     return ",".join(tiers_applied), paths

# def _ensure_dir(p):
#     if p: os.makedirs(p, exist_ok=True)

# def _run(cmd, dry):
#     if dry:
#         print("$", " ".join(shlex.quote(c) for c in cmd))
#         return 0
#     return subprocess.call(cmd)

# def _normalize_log_path(path):
#     if not path: return None
#     if os.path.isdir(path):
#         return os.path.join(path, "export_log.parquet")
#     return path

# # -------- main API --------

# def build_export_plan(
#     filtered_table: pa.Table,
#     alias_csv: str = None,
#     include_visuals: bool = False,
#     include_videos: bool = True
# ):
#     alias_map = load_aliases(alias_csv)
#     plan = []
#     for r in filtered_table.to_pylist():
#         src = _to_str(r.get("rec_path"))
#         if not src: continue

#         is_social = _truthy(r.get("social"))
#         yymmdd, hms = _extract_date_time(r)
#         sid = _pick_id(r, alias_map, is_social)
#         side = "social" if is_social else "single"

#         # Layout: <single|social>/<YY_MM_DD>/<ID>_<HHMMSS>/
#         dest_rel = _join(side, yymmdd, f"{sid}_{hms}")

#         tiers_applied, rel_paths = _assign_tiers(r, include_visuals=include_visuals, include_videos=include_videos)
#         plan.append({
#             "source_rec_path": src,
#             "dest_rel_path": dest_rel,
#             "tiers_applied": tiers_applied,
#             "tier_paths": rel_paths,
#             "is_social": is_social,
#             "id": sid,
#             "yymmdd": yymmdd,
#             "hms": hms,
#         })
#     return plan

# def export_with_rsync(
#     plan,
#     export_root,
#     dry_run=True,
#     mode="staged",
#     dataset_version=None,
#     profile_label="default",
#     log_parquet_path=None,
#     log_on_dry_run=False
# ):
#     if mode not in {"staged","snapshot"}:
#         raise ValueError("mode must be 'staged' or 'snapshot'")

#     root = _join(export_root, f"v{dataset_version}") if (mode=="snapshot" and dataset_version) else export_root

#     log_rows = []
#     for item in plan:
#         src_root = item["source_rec_path"]
#         dest = _join(root, item["dest_rel_path"])

#         if not dry_run:
#             _ensure_dir(dest)

#         for rel in item["tier_paths"]:
#             src_glob = _join(src_root, rel)
#             matches = sorted(glob.glob(src_glob))
#             if not matches:
#                 continue
#             for src in matches:
#                 cmd = ["rsync", "-a"]
#                 if dry_run: cmd.append("-n")
#                 cmd += [src, dest + "/"]
#                 _run(cmd, dry_run)

#         log_rows.append({
#             "export_time_utc": dt.datetime.utcnow().isoformat(timespec="seconds"),
#             "profile_label": profile_label,
#             "export_mode": mode,
#             "dataset_version": dataset_version or "",
#             "source_rec_path": src_root,
#             "dest_rel_path": item["dest_rel_path"],
#             "tiers_applied": item["tiers_applied"],
#             "dry_run": bool(dry_run),
#         })

#     if log_parquet_path and (not dry_run or log_on_dry_run):
#         log_path = _normalize_log_path(_to_str(log_parquet_path))
#         parent = os.path.dirname(log_path)
#         _ensure_dir(parent or ".")
#         _append_parquet(log_path, pa.Table.from_pylist(log_rows))

# def _append_parquet(path, new_table: pa.Table):
#     if os.path.exists(path):
#         old = pq.read_table(path)
#         out = pa.concat_tables([old, new_table], promote=True)
#     else:
#         out = new_table
#     pq.write_table(out, path)


In [4]:
# # 0) you already have filtered_table from your Arrow filter
# # from export_plan_min import build_export_plan, export_with_rsync

# # 1) build plan (optional alias CSV if you have it)
# animal_aliases = "./animal_aliases.csv"
# plan = build_export_plan(filtered_table, alias_csv=animal_aliases)

# # 2) dry-run, staged
# export_with_rsync(
#     plan,
#     export_root="/data/big_rim/sorted_mir_dataset/251023_test_socical_10",
#     dry_run=True,
#     mode="staged",                 # or "snapshot"
#     dataset_version=None,          # e.g., "2025-10-23" if snapshot
#     profile_label="251023_social_test", #sync1_com1_sep_0910_0912
#     log_parquet_path="/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs",
#     log_on_dry_run=False
# )

# # 3) real run
# # export_with_rsync(
# #     plan,
# #     export_root="/path/to/EXPORT_ROOT",
# #     dry_run=False,
# #     mode="staged",
# #     dataset_version=None,
# #     profile_label="sync1_com1_sep_0910_0912",
# #     log_parquet_path="/path/to/export_log.parquet",
# # )


# profile.py example, with yaml input

In [5]:
from export_profiles import load_profiles, build_export_plan, export_with_rsync

profiles = load_profiles("profiles.yaml")
profile  = profiles["paper-core"]   # pick one: paper-core | qc-review | dannce-delta | repro-full

# Build plan (IDs via stronger session parsing + aliases; date as YYYY_MM_DD)
plan = build_export_plan(
    filtered_table,
    alias_csv="./animal_aliases.csv",   # optional but recommended
    profile=profile
)

# Dry run (prints commands only)
export_with_rsync(
    plan,
    export_root="/data/big_rim/sorted_mir_dataset/251023_test_social_10",
    dry_run=False,
    mode="staged",
    profile_label="paper-core",
    log_parquet_path="/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs"  # dir ok
)

# # Real run
# export_with_rsync(
#     plan,s
#     export_root="/data/big_rim/sorted_mir_dataset/251023_test_social_10",
#     dry_run=False,ss
#     mode="staged",  # or "snapshot" with dataset_version="2025-10-23"
#     profile_label="paper-core",
#     log_parquet_path="/home/lq53/mir_repos/BBOP/random_tests/25oct_dataset_prep/export_logs"
# )
